In [1]:
#file to compare the results of the long unique version, and the separate initial scripts

import os
import json
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from collections.abc import Mapping, Sequence

# ------------------------------------------------------------
# Helper functions (unchanged)
# ------------------------------------------------------------
def _dicts_equal(d1, d2):
    """Recursively compare two nested structures, ignoring key order in dicts."""
    if type(d1) != type(d2):
        return False
    if isinstance(d1, Mapping):
        if set(d1.keys()) != set(d2.keys()):
            return False
        return all(_dicts_equal(d1[k], d2[k]) for k in d1)
    if isinstance(d1, Sequence) and not isinstance(d1, (str, bytes)):
        if len(d1) != len(d2):
            return False
        return all(_dicts_equal(i, j) for i, j in zip(d1, d2))
    return d1 == d2


def compare_tif(path1, path2):
    with rasterio.open(path1) as src1, rasterio.open(path2) as src2:
        # Metadati essenziali
        if src1.count != src2.count:
            return False, f"band count differ: {src1.count} vs {src2.count}"
        if src1.shape != src2.shape:
            return False, f"shape differ: {src1.shape} vs {src2.shape}"
        if src1.dtypes[0] != src2.dtypes[0]:
            return False, f"dtype differ: {src1.dtypes[0]} vs {src2.dtypes[0]}"
        if src1.crs != src2.crs:
            return False, f"CRS differ: {src1.crs} vs {src2.crs}"
        # Confronta tutte le bande
        for b in range(1, src1.count + 1):
            arr1 = src1.read(b)
            arr2 = src2.read(b)
            if not np.array_equal(arr1, arr2):
                diff_count = np.count_nonzero(arr1 != arr2)
                return False, f"band {b} differs ({diff_count} different pixels)"
        return True, "OK"


def compare_gpkg(path1, path2):
    """Compare all layers of two GeoPackages (row‑wise, order‑independent)."""
    layers1 = gpd.list_layers(path1)
    layers2 = gpd.list_layers(path2)
    if len(layers1) != len(layers2):
        return False, f"different number of layers ({len(layers1)} vs {len(layers2)})"

    for layer_name in layers1['name']:
        if layer_name not in layers2['name'].values:
            return False, f"layer '{layer_name}' missing in second file"
        gdf1 = gpd.read_file(path1, layer=layer_name)
        gdf2 = gpd.read_file(path2, layer=layer_name)
        if gdf1.shape != gdf2.shape:
            return False, f"layer '{layer_name}': shape {gdf1.shape} vs {gdf2.shape}"

        sort_cols = list(gdf1.columns)
        gdf1['_geom_wkt'] = gdf1.geometry.to_wkt()
        gdf2['_geom_wkt'] = gdf2.geometry.to_wkt()
        sort_cols_with_geom = sort_cols + ['_geom_wkt']

        gdf1_sorted = gdf1.sort_values(by=sort_cols).reset_index(drop=True)
        gdf2_sorted = gdf2.sort_values(by=sort_cols).reset_index(drop=True)

        gdf1_sorted = gdf1_sorted.drop(columns=['_geom_wkt'])
        gdf2_sorted = gdf2_sorted.drop(columns=['_geom_wkt'])

        try:
            pd.testing.assert_frame_equal(gdf1_sorted, gdf2_sorted, check_like=True, check_dtype=False)
        except AssertionError:
            return False, f"layer '{layer_name}': dataframes differ"
    return True, "OK"


def compare_csv(path1, path2):
    """Compare two CSV files as DataFrames (row‑order independent)."""
    df1 = pd.read_csv(path1)
    df2 = pd.read_csv(path2)
    if df1.shape != df2.shape:
        return False, f"shape {df1.shape} vs {df2.shape}"
    cols = list(df1.columns)
    df1_sorted = df1.sort_values(by=cols).reset_index(drop=True)
    df2_sorted = df2.sort_values(by=cols).reset_index(drop=True)
    try:
        pd.testing.assert_frame_equal(df1_sorted, df2_sorted, check_dtype=False)
        return True, "OK"
    except AssertionError:
        return False, "dataframes differ"


def compare_json(path1, path2):
    """Compare two JSON files structurally (order‑independent on keys)."""
    with open(path1, 'r', encoding='utf-8') as f1:
        data1 = json.load(f1)
    with open(path2, 'r', encoding='utf-8') as f2:
        data2 = json.load(f2)
    if _dicts_equal(data1, data2):
        return True, "OK"
    else:
        return False, "JSON structures differ"


# ------------------------------------------------------------
# Main comparison loop (updated for Jupyter / interactive use)
# ------------------------------------------------------------

# Try to get the directory of this script (works when run as .py file).
# If __file__ is not defined (Jupyter, interactive console), use current working directory.
try:
    base_dir = os.path.dirname(os.path.abspath(__file__))
except NameError:
    base_dir = os.getcwd()
    print(f"⚠ '__file__' not defined. Using current working directory as base:\n   {base_dir}\n"
          "   Make sure this is the 'script_nostri' folder.\n")

folder1 = os.path.join(base_dir, 'dataset_big2')
folder2 = os.path.join(base_dir, 'dataset_big')

COMPARATORS = {
    '.tif':  compare_tif,
    '.tiff': compare_tif,
    '.gpkg': compare_gpkg,
    '.csv':  compare_csv,
    '.json': compare_json,
}

try:
    files = [f for f in os.listdir(folder1)
             if os.path.isfile(os.path.join(folder1, f))]
except FileNotFoundError:
    print(f"ERROR: folder 'dataset_big2' not found in {base_dir}")
    exit(1)

if not os.path.exists(folder2):
    print(f"ERROR: folder 'dataset_big' not found in {base_dir}")
    exit(1)

files.sort()
print("Data‑aware comparison between dataset_big2 and dataset_big:\n")

all_ok = True

for filename in files:
    path1 = os.path.join(folder1, filename)
    path2 = os.path.join(folder2, filename)

    if not os.path.exists(path2):
        print(f"MISSING: '{filename}' not in dataset_big")
        all_ok = False
        continue

    ext = os.path.splitext(filename)[1].lower()
    comparator = COMPARATORS.get(ext)
    if comparator is None:
        import filecmp
        if filecmp.cmp(path1, path2, shallow=False):
            print(f"OK (binary): '{filename}' identical")
        else:
            print(f"DIFFERENT (binary): '{filename}' differs")
            all_ok = False
        continue

    try:
        identical, msg = comparator(path1, path2)
    except Exception as e:
        print(f"ERROR processing '{filename}': {e}")
        all_ok = False
        continue

    if identical:
        print(f"OK: '{filename}' identical")
    else:
        print(f"DIFFERENT: '{filename}' — {msg}")
        all_ok = False

if all_ok:
    print("\nAll files contain the same data.")
else:
    print("\nSome files differ or are missing. Check the details above.")

⚠ '__file__' not defined. Using current working directory as base:
   c:\Users\matti\OneDrive - Politecnico di Milano\Thesis_onstoveM&F\OnStoveThesis\thesis\script_nostri
   Make sure this is the 'script_nostri' folder.

Data‑aware comparison between dataset_big2 and dataset_big:

OK: 'CO_SSA_avg.tif' identical
OK (binary): 'Country_boundaries.geojson' identical
OK: 'NOx_SSA_avg.tif' identical
OK: 'Population.tif' identical
OK: 'Urban.tif' identical
DIFFERENT: 'chain_with_cost.gpkg' — layer 'filling': dataframes differ
OK: 'depot_interim.json' identical
DIFFERENT: 'end_user_price.tif' — band 9 differs (559129 different pixels)
OK: 'filling_interim.json' identical
OK: 'filling_point_clients.gpkg' identical
OK: 'first_step.gpkg' identical
OK: 'friction_moto.tif' identical
OK: 'friction_walk.tif' identical
OK: 'full_lpg_chain_nig_3857-Francesco.gpkg' identical
OK: 'full_lpg_chain_nig_3857.gpkg' identical
OK: 'huff_preferred_distributor_per_pixel.tif' identical
OK: 'huff_reseller_lookup.cs

In [2]:
import json
import os

base = r'C:\Users\matti\OneDrive - Politecnico di Milano\Thesis_onstoveM&F\OnStoveThesis\thesis\script_nostri'
with open(os.path.join(base, 'dataset_big2', 'huff_run_profile.json')) as f1, \
     open(os.path.join(base, 'dataset_big',  'huff_run_profile.json')) as f2:
    data1 = json.load(f1)
    data2 = json.load(f2)

# Deep equality check (already done), but let's find the exact differing keys
def diff_dicts(d1, d2, path=""):
    if type(d1) != type(d2):
        print(f"Type mismatch at {path}: {type(d1).__name__} vs {type(d2).__name__}")
        return
    if isinstance(d1, dict):
        for k in d1.keys() | d2.keys():
            if k not in d1:
                print(f"Missing key '{k}' in first file at {path}")
            elif k not in d2:
                print(f"Missing key '{k}' in second file at {path}")
            else:
                diff_dicts(d1[k], d2[k], f"{path}.{k}" if path else k)
    elif isinstance(d1, list):
        if len(d1) != len(d2):
            print(f"List length differs at {path}: {len(d1)} vs {len(d2)}")
        else:
            for i, (a, b) in enumerate(zip(d1, d2)):
                diff_dicts(a, b, f"{path}[{i}]")
    else:
        if d1 != d2:
            print(f"Value differs at {path}: {d1!r} vs {d2!r}")

diff_dicts(data1, data2)

Value differs at run_date: '2026-05-13 09:43:47' vs '2026-05-13 11:04:07'
Value differs at inputs.dataset_dir: 'dataset_big2' vs 'dataset_big'
Value differs at timings_seconds.walk_solver_phase1: 16.398393100011162 vs 15.163340899976902
Value differs at timings_seconds.moto_solver_phase1: 15.834261399984825 vs 15.771419600001536
Value differs at timings_seconds.total: 632.1041505999747 vs 420.6819669000106
Value differs at timings_seconds.read_population: 0.024482699984218925 vs 0.0235192000400275
Value differs at timings_seconds.connected_components: 0.028754599974490702 vs 0.017610799986869097
Value differs at timings_seconds.reproject: 0.35906589997466654 vs 0.2839440000243485
Value differs at timings_seconds.exact_recomputation: 597.6851024000207 vs 387.7469556000433
Value differs at timings_seconds.load_seeds: 0.1304029999882914 vs 0.06801200000336394
